# Info

This script performs a Random Forest classification.

The following parameters can be specified:

#### TRAIN_DATA:

The dataset the random forest will be trained on. \

N22 = Northern Bavaria 2022\
N23 = Northern Bavaria 2023\
S20 = Southern Bavaria 2020\
S22 = Southern Bavaria 2022\
ALL = all of the above\

#### VAL_DATA:

The dataset the random forest will be validated on.\

N22 = Northern Bavaria 2022\
N23 = Northern Bavaria 2023\
S20 = Southern Bavaria 2020\
S22 = Southern Bavaria 2022\
ALL = all of the above\

_If VAL_DATA == TRAIN_DATA, as 5-fold Cross Validation will be used._

#### method:

This is the sampling method:\

weighted: Solely minority class ("deadwood") gets oversampled. Specify **weight** as well. Specify **total_n** (number of observations for the model) as well.\
undersampling: All the classes get undersampled to the number of the minority class.\
oversampling: All the classes get oversampled to the number of the majority class ("undisturbed"). Specify, XXXXx as well.




# 1 Import packages and functions

In [ ]:
import xarray as xr
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold

from RF_TRAINING import chosen_bands

In [ ]:
from plot_functions import *
from read_functions import *
from datahandling_functions import *
from analysis_functions import *

# 2 Read

In [ ]:
chosen_bands =
TRAIN_DATA =
VAL_DATA =
method = "weighted"
# if method = weight:
weight = 4


## 2.1 Read Train

In [ ]:
if TRAIN_DATA == "ALL":

    df1 = open_to_pd_df_withregionlabels(r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_withindices\n22.tif")
    df2 = open_to_pd_df_withregionlabels(r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_withindices\n23.tif")
    df3 = open_to_pd_df_withregionlabels(r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_withindices\s20.tif")
    df4 = open_to_pd_df_withregionlabels(r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_withindices\s22.tif")
    df = pd.concat([df1, df2, df3, df4], axis=0, ignore_index=True)
    df = df.reset_index()



if TRAIN_DATA != "ALL":

    df = open_to_pd_df(f"C:/Users/miles/OneDrive/Dokumente/ROOT/trainingdata_collection/trainingdata_withindices/{TRAIN_DATA.lower()}.tif")

df.columns.name = None
df = df.loc[:, ~df.columns.duplicated()]
final_selection = ["x", "y"] + [b for b in chosen_bands if b not in ["x", "y", "trainclass"]] + ["trainclass"]
train_df = df[final_selection]

## 2.2 Read Val


In [ ]:
if VAL_DATA != TRAIN_DATA:

    val_df = open_to_pd_df(f"C:/Users/miles/OneDrive/Dokumente/ROOT/trainingdata_collection/trainingdata_withindices/{VAL_DATA.lower()}.tif")
    val_df = val_df.reset_index()
    val_df.columns.name = None
    val_df = val_df.loc[:, ~val_df.columns.duplicated()]
    final_selection = ["x", "y"] + [b for b in chosen_bands if b not in ["x", "y", "trainclass"]] + ["trainclass"]
    val_df = val_df[final_selection]

    forestclass_test = val_df["trainclass"]
    pred_test = val_df.drop(columns=["trainclass", "region", "regionclass"])

    forestclass_train = train_df["trainclass"]
    pred_train = train_df.drop(columns=["trainclass", "region", "regionclass"])


if VAL_DATA == TRAIN_DATA:

    df = rf_sample(train_df, method = method, weight = weight)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    pred = df.drop(columns=['trainclass', 'region', 'region_class'])
    forestclass = df['trainclass']
    strat_col = df['region_class']

# 3. TRAIN

In [ ]:
if VAL_DATA == TRAIN_DATA:

    all_accuracies = []
    all_f1s = []
    all_cms = []
    cv_predictions = np.zeros_like(y)

    print(f"{'Fold':<10} | {'Accuracy':<10} | {'F1 (Weighted)':<15}")
    print("-" * 45)

    for i, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        x_train, x_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        rf = randomForestClass(ntrees = 750, pred_train=x_train, forestclass_train=y_train, FIT = FIT)
        preds = rf.predict(x_test)
        cv_predictions[test_idx] = preds
        acc = accuracy_score(y_test, preds)

if VAL_DATA != TRAIN_DATA:

    rf = randomForestClass(ntrees = 750, pred_train=pred_train, forestclass_train=forestclass_train)